# Projeto de Parceria | Semantix — SVM x XGBoost

## Classificação de tumores de mama

Objetivo: comparar SVM e XGBoost no Breast Cancer Wisconsin Diagnostic Dataset. Este é um estudo acadêmico de Machine Learning e não uma ferramenta de diagnóstico médico.


## 1. Coleta de dados
O conjunto Breast Cancer Wisconsin (Diagnostic) é público e está disponível pela UCI Machine Learning Repository e pelo `scikit-learn`. Nesta versão, usamos `load_breast_cancer`, que fornece 569 amostras, 30 características numéricas e a variável alvo (0 = maligno; 1 = benigno).


In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
print(df.shape)
print('Valores ausentes:', df.isna().sum().sum())
df.head()


## 2. Análise exploratória
Verificamos dimensões, valores ausentes, estatísticas descritivas e distribuição das classes.


In [ ]:
print(df['target'].value_counts())
df.describe().T


## 3. Preparação e divisão dos dados
Usamos 80% para treino e 20% para teste, com estratificação. No SVM, a padronização é feita dentro de um `Pipeline`, evitando vazamento de dados durante a validação cruzada.


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
X = df.drop(columns='target'); y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 4. Modelagem — SVM
O SVM foi combinado com `StandardScaler` e otimizado por `GridSearchCV`, usando ROC-AUC como métrica de seleção.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
svm_pipe = Pipeline([('scaler', StandardScaler()),('svc', SVC(probability=True, random_state=42))])
svm_grid = {'svc__C':[0.1,1,10,100], 'svc__gamma':['scale',0.001,0.01,0.1], 'svc__kernel':['rbf']}
svm_search = GridSearchCV(svm_pipe, svm_grid, scoring='roc_auc', cv=cv, n_jobs=-1)
svm_search.fit(X_train,y_train)
svm_search.best_params_


## 5. Modelagem — XGBoost
O XGBoost foi otimizado separadamente, avaliando profundidade, taxa de aprendizado, número de árvores e amostragem.


In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier(objective='binary:logistic',eval_metric='logloss',random_state=42,n_jobs=1,tree_method='hist')
xgb_grid = {'n_estimators':[100,200], 'max_depth':[2,3,4], 'learning_rate':[0.03,0.05,0.1], 'subsample':[0.8,1.0], 'colsample_bytree':[0.8,1.0]}
xgb_search = GridSearchCV(xgb, xgb_grid, scoring='roc_auc', cv=cv, n_jobs=-1)
xgb_search.fit(X_train,y_train)
xgb_search.best_params_


## 6. Avaliação e visualização
Comparamos Accuracy, Precision, Recall, F1 e ROC-AUC no conjunto de teste, além de matrizes de confusão e curva ROC.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
resultados=[]
for nome,modelo in {'SVM':svm_search.best_estimator_,'XGBoost':xgb_search.best_estimator_}.items():
    pred=modelo.predict(X_test); prob=modelo.predict_proba(X_test)[:,1]
    resultados.append([nome,accuracy_score(y_test,pred),precision_score(y_test,pred),recall_score(y_test,pred),f1_score(y_test,pred),roc_auc_score(y_test,prob)])
pd.DataFrame(resultados,columns=['Modelo','Accuracy','Precision','Recall','F1','ROC_AUC'])


## 7. Resultados obtidos

| Modelo | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| SVM | 0.9825 | 0.9861 | 0.9861 | 0.9861 | 0.9977 |
| XGBoost | 0.9561 | 0.9467 | 0.9861 | 0.9660 | 0.9937 |

Melhores parâmetros SVM: `{'svc__C': 10, 'svc__gamma': 0.01, 'svc__kernel': 'rbf'}`.

Melhores parâmetros XGBoost: `{'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 200, 'subsample': 0.8}`.


## 8. Conclusões
Neste experimento, o SVM apresentou o melhor desempenho geral no conjunto de teste, especialmente em Accuracy, F1 e ROC-AUC. O XGBoost também apresentou desempenho elevado e Recall equivalente. Os resultados reforçam que a escolha do algoritmo deve ser orientada por validação experimental, e não pela suposição de que um método mais complexo será necessariamente superior.

Como limitação, o conjunto é relativamente pequeno. Uma continuação natural seria repetir a avaliação com validação cruzada aninhada, testar calibração e avaliar outros conjuntos de dados.

**Observação:** este trabalho tem finalidade acadêmica e não deve ser utilizado para diagnóstico médico.
